# SpellBreaker on IBM LinuxONE

This notebook records a live scoring run against the service on the LinuxONE Community Cloud VM. The transfer below is synthetic demo data; no real customer information is used.

In [1]:
import importlib.metadata
import json
import platform
import time
from urllib.request import Request, urlopen

BASE_URL = 'http://127.0.0.1:8000'
def get_json(path):
    with urlopen(BASE_URL + path, timeout=10) as response:
        return json.loads(response.read())

health = get_json('/health')
report = get_json('/report')
assert platform.machine() == 's390x', platform.machine()
assert health['arch'] == 's390x' and health['trained_on'] == 's390x', health
assert report['trained_on'] == 's390x', report.get('trained_on')
packages = {name: importlib.metadata.version(name) for name in ('numpy', 'pandas', 'scikit-learn', 'scipy', 'joblib')}
print(json.dumps({'python': platform.python_version(), 'machine': platform.machine(), 'health': health, 'packages': packages}, indent=2))

{
  "python": "3.12.15",
  "machine": "s390x",
  "health": {
    "status": "ok",
    "model_version": "0.1.0",
    "arch": "s390x",
    "trained_on": "s390x",
    "genai": "template"
  },
  "packages": {
    "numpy": "2.0.2",
    "pandas": "3.0.6",
    "scikit-learn": "1.9.0",
    "scipy": "1.13.1",
    "joblib": "1.6.0"
  }
}


In [2]:
reset_req = Request(BASE_URL + '/demo/reset', data=b'{}', headers={'Content-Type': 'application/json'}, method='POST')
with urlopen(reset_req, timeout=10) as response:
    print('demo_reset:', response.status)
payload = {
    'customer_id': 'C-TAN',
    'payee_account': 'MB-778-201-9931',
    'payee_name': 'Secure Govt Account',
    'amount': 20000,
    'source_pot': 'savings',
    'flags': {'on_call': True, 'pasted_account': True, 'rushed_score': 0.8}
}
score_req = Request(BASE_URL + '/score', data=json.dumps(payload).encode(), headers={'Content-Type': 'application/json'}, method='POST')
with urlopen(score_req, timeout=10) as response:
    decision = json.loads(response.read())
assert decision['level'] == 'CRITICAL', decision
print(json.dumps({key: decision[key] for key in ('decision_id', 'risk', 'level', 'scam_type', 'action', 'reasons', 'latency_ms', 'scored_on')}, indent=2))

demo_reset: 200
{
  "decision_id": "d_000001",
  "risk": 0.9990560309085813,
  "level": "CRITICAL",
  "scam_type": "gov_impersonation",
  "action": "HOLD_AND_ESCALATE",
  "reasons": [
    "66.7\u00d7 your usual transfer amount",
    "Receiving account shows money-mule behaviour (0.91)",
    "You're on an active phone call"
  ],
  "latency_ms": 5.60520799990627,
  "scored_on": "IBM LinuxONE (s390x)"
}


In [3]:
case = None
for _ in range(120):
    case = get_json('/case/' + decision['decision_id'])
    if case.get('status') == 'ready':
        break
    time.sleep(0.25)
assert case and case.get('status') == 'ready', case
print(json.dumps({'case_runtime': case.get('runtime'), 'case_status': case.get('status'), 'case_note': case.get('note')}, indent=2))

{
  "case_runtime": "template",
  "case_status": "ready",
  "case_note": "A critical risk transfer of S$10k\u201350k was detected. Indicators include 66.7\u00d7 your usual transfer amount, Receiving account shows money-mule behaviour (0.91), You're on an active phone call. The likely typology is gov impersonation. Call the customer using the bank's registered number; ask them to end any active call and independently verify the recipient. If the customer insists and coercion remains apparent, consider whether a Protection from Scams Act escalation is warranted."
}


## Training snapshot

The model report served by the LinuxONE service includes temporal test metrics, age-band fairness, per-scam-type recall, ablations, and leave-one-type-out recall. Scores describe the synthetic simulator and are not a real-world effectiveness estimate.

In [4]:
print(json.dumps({
    'trained_on': report['trained_on'],
    'difficulty': report.get('difficulty'),
    'generation_parameters': report.get('generation_parameters'),
    'full_metrics': report['full'],
    'fairness_by_age_band': report['fairness_by_age_band'],
}, indent=2))

{
  "trained_on": "s390x",
  "difficulty": null,
  "generation_parameters": {
    "difficulty": "realistic",
    "customers": 20000,
    "rows": 100000,
    "scam_fraction": 0.07455,
    "hard_negative_rate": 0.04,
    "behavioral_label_noise": 0.03,
    "overlap_adjustment": "on_call scam .24 (gov .72), pasted gov .60, new payee scam .36 (gov .75), 10% legit high-balance transfers; 450 gov, 300 job-sequence and 300 rising-investment training-only typology cases",
    "seed": 42
  },
  "full_metrics": {
    "pr_auc": 0.8805388735560538,
    "roc_auc": 0.9707501826144027,
    "recall_at_t1": 0.7759699624530664,
    "precision_at_t1": 0.8458390177353342,
    "legit_interruption_rate": 0.011386537686416768
  },
  "fairness_by_age_band": {
    "18-34": {
      "legit_interruption_rate": 0.010941644562334218,
      "scam_recall": 0.7427884615384616,
      "n_legit": 6032,
      "n_scam": 416
    },
    "35-49": {
      "legit_interruption_rate": 0.012414965986394558,
      "scam_recall": 0.

## Notebook-only PyTorch MLP challenger
This small temporal holdout comparison uses the synthetic SpellBreaker simulator. It is an experiment only: the deployed M1 remains scikit-learn HistGradientBoosting, and this MLP is not served.

In [5]:
import numpy as np
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from ml.spellbreaker_train import FEATS, synthesize

torch.set_num_threads(1)
torch.manual_seed(42)
challenger_df = synthesize(customers=1200, difficulty="realistic", seed=77)
train_mask = challenger_df.day <= 64
test_mask = challenger_df.day >= 77
X_train = challenger_df.loc[train_mask, FEATS].to_numpy(dtype=np.float32)
y_train = challenger_df.loc[train_mask, "label"].to_numpy(dtype=np.float32)
X_test = challenger_df.loc[test_mask, FEATS].to_numpy(dtype=np.float32)
y_test = challenger_df.loc[test_mask, "label"].to_numpy(dtype=np.int64)

hist = HistGradientBoostingClassifier(max_iter=80, max_leaf_nodes=12, random_state=42)
hist.fit(X_train, y_train.astype(np.int64))
hist_prob = hist.predict_proba(X_test)[:, 1]

mlp = nn.Sequential(nn.Linear(len(FEATS), 24), nn.ReLU(), nn.Linear(24, 8), nn.ReLU(), nn.Linear(8, 1))
optimizer = torch.optim.Adam(mlp.parameters(), lr=0.003)
loss_fn = nn.BCEWithLogitsLoss()
loader = DataLoader(TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)), batch_size=256, shuffle=True)
for _ in range(25):
    for xb, yb in loader:
        optimizer.zero_grad(set_to_none=True)
        loss = loss_fn(mlp(xb).squeeze(1), yb)
        loss.backward()
        optimizer.step()
with torch.no_grad():
    mlp_prob = torch.sigmoid(mlp(torch.from_numpy(X_test)).squeeze(1)).numpy()
challenger_report = {
    "scope": "notebook-only synthetic challenger; not deployed",
    "trained_on": platform.machine(),
    "train_rows": int(train_mask.sum()),
    "test_rows": int(test_mask.sum()),
    "models": {
        "scikit_learn_histgradientboosting_m1": {
            "pr_auc": float(average_precision_score(y_test, hist_prob)),
            "roc_auc": float(roc_auc_score(y_test, hist_prob)),
        },
        "pytorch_mlp_challenger": {
            "pr_auc": float(average_precision_score(y_test, mlp_prob)),
            "roc_auc": float(roc_auc_score(y_test, mlp_prob)),
            "epochs": 25,
            "hidden_layers": [24, 8],
        },
    },
}
print(json.dumps(challenger_report, indent=2))


{
  "scope": "notebook-only synthetic challenger; not deployed",
  "trained_on": "s390x",
  "train_rows": 3492,
  "test_rows": 1302,
  "models": {
    "scikit_learn_histgradientboosting_m1": {
      "pr_auc": 0.8678815998467194,
      "roc_auc": 0.9800316678395496
    },
    "pytorch_mlp_challenger": {
      "pr_auc": 0.8443928013021668,
      "roc_auc": 0.9752326217843459,
      "epochs": 25,
      "hidden_layers": [
        24,
        8
      ]
    }
  }
}
